In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, sum as spark_sum, avg, when, round
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DoubleType

spark = SparkSession.builder.appName("RollingRevenue").getOrCreate()

# Dummy data: user_id, created_at, purchase_amt (positive = purchase, negative = return)
data = [
    (1, "2020-01-05", 100),
    (2, "2020-01-15", 50),
    (3, "2020-01-20", 75),
    (1, "2020-02-10", 200),
    (2, "2020-02-14", -30),   # return (negative)
    (4, "2020-02-18", 120),
    (1, "2020-03-05", 80),
    (3, "2020-03-12", 60),
    (5, "2020-03-25", 150),
    (2, "2020-04-03", 90),
    (4, "2020-04-17", 110),
    (1, "2020-05-01", 130),
    (3, "2020-05-20", 70),
    (5, "2020-05-28", 95),
    (2, "2020-06-10", 85),
]

# Create DataFrame
df = spark.createDataFrame(data, ["user_id", "created_at", "purchase_amt"])
df = df.withColumn("created_at", to_date(col("created_at")))

df.createOrReplaceTempView("amazon_purchases")

# Preview
df.show()

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
spark.sql("""
with monthly_revenue as (
  SELECT 
    DATE_FORMAT(created_at, 'yyyy-MM') AS year_month,
    SUM(CASE WHEN purchase_amt > 0 THEN purchase_amt ELSE 0 END) AS total_revenue
  FROM amazon_purchases
  WHERE purchase_amt > 0  -- Exclude returns/negative values
  GROUP BY DATE_FORMAT(created_at, 'yyyy-MM'))

select 
    year_month, 
    total_revenue,
    AVG(total_revenue) OVER(order by year_month rows between 2 preceding and current row) as rolling_avg
from monthly_revenue

""").display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when, date_format, round
from pyspark.sql.window import Window

# Step 1: Filter out returns (negative purchase_amt)
filtered_df = df.filter(col("purchase_amt") > 0)

# Step 2: Aggregate revenue by year-month
monthly_revenue = filtered_df.groupBy(
    date_format("created_at", "yyyy-MM").alias("year_month")
).agg(
    spark_sum("purchase_amt").alias("total_revenue")
).orderBy("year_month")

# Step 3: Calculate 3-month rolling average
window_spec = Window.orderBy("year_month").rowsBetween(-2, 0)

rolling_revenue = monthly_revenue.withColumn(
    "rolling_3_month_avg",
    round(avg("total_revenue").over(window_spec), 2)
)

# Step 4: Show results
rolling_revenue.select("year_month", "rolling_3_month_avg").show()